# 4. Hafta — Conway'in Yaşam Oyunu ve Beliren Nesneler

**Proje:** Hücresel Otomatlarda Beliren Karmaşıklık  
**Not defteri:** `04_game_of_life.ipynb`  
**Tahmini süre:** 8–10 saat

İlk üç haftada tek boyutlu hücresel otomatlarda yerel kuralları, sınıflandırmayı ve pertürbasyon yayılmasını inceledin. Bu hafta ikinci bir uzay boyutu ekliyoruz.

Ana araştırma sorumuz:

> **Sabit yerlerde duran ve yalnızca doğup ölen hücreler, nasıl parçacık gibi hareket eden kalıcı nesneler oluşturabilir?**

Bu soru projenin genel temasının merkezindedir: mikroskobik kurallarda açıkça bulunmayan yeni bir betimleme, daha büyük ölçekte **belirebilir**.

## Önceki haftalardan bu haftaya

- **1. hafta:** Yerel bir kuralın başlangıç koşulunu büyük ölçekli bir desene dönüştürdüğünü gördün.
- **2. hafta:** Görsel desenleri yoğunluk ve etkinlik gibi gözlenebilirlerle destekledin; tek bir sayının uzamsal yapıyı tamamen açıklamadığını öğrendin.
- **3. hafta:** Tek hücrelik bilginin kaybolabileceğini, yerel kalabileceğini veya bir ışık konisi içinde yayılabileceğini ölçtün.
- **4. hafta:** Artık uzamsal yapıların kendisini inceleyeceğiz: sabit nesneler, salınıcılar ve hareket eden gliderlar.

İki boyut sayesinde bir desenin yalnızca büyümesi değil, biçimini koruyarak **yer değiştirmesi** mümkün olur. Bu, ileride rastgele başlangıç koşullarını ve topluluk istatistiklerini inceleyeceğimiz 5. haftanın da temelini oluşturacaktır.

### Öğrenme hedefleri

Bu not defterinin sonunda şunları yapabilmelisin:

1. sekiz komşuyu periyodik sınırlarla saymak;
2. Game of Life'ın B3/S23 kuralını uygulamak;
3. iki boyutlu bir hücresel otomatı benzetmek;
4. still life, oscillator ve glider örneklerini yeniden üretmek;
5. nüfus ve periyot ölçmek;
6. bir gliderın yer değiştirmesini ve etkin hızını hesaplamak;
7. gliderın neden beliren bir nesne olduğunu açıklamak.

## Model: B3/S23

Her hücrenin durumu

\[
s_{ij}(t)\in\{0,1\}
\]

olsun. Hücre \((i,j)\), çevresindeki sekiz hücreyle etkileşir. Merkez hücre komşu sayısına dâhil edilmez.

Bir sonraki adımda:

- **Doğum (B3):** Ölü hücre, tam olarak 3 canlı komşusu varsa doğar.
- **Hayatta kalma (S23):** Canlı hücre, 2 veya 3 canlı komşusu varsa yaşar.
- Diğer bütün durumlarda hücre ölür veya ölü kalır.

Bu kural kısaca **B3/S23** biçiminde yazılır.

### Çalıştırmadan önce tahmin et

1. Tek başına duran canlı hücreye ne olur?
2. Tamamen dolu büyük bir bölgenin iç kısmındaki hücrelere ne olur?
3. Bir desen hücreleri fiziksel olarak taşımadan hareket edebilir mi?

**Tahminlerim:**

> Buraya 3–5 cümle yaz.

<details>
<summary>İpucu: İlk iki soruda komşuları say</summary>

Tek hücrenin canlı komşusu yoktur ve yalnızlıktan ölür. Tam dolu bir bölgenin iç hücresinin sekiz canlı komşusu vardır ve aşırı kalabalıktan ölür.
</details>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams["figure.dpi"] = 120
plt.rcParams["image.cmap"] = "binary"


def check(label, test_function):
    '''Bir öğrenci testini not defterini durdurmadan çalıştır.'''
    try:
        test_function()
    except (AssertionError, NotImplementedError) as error:
        message = str(error) or "sonuç beklenen değerle eşleşmedi"
        print(f"○ {label}: çalışmaya devam et — {message}")
    else:
        print(f"✓ {label}: geçti")


print("Kurulum tamamlandı.")

## 1. Sekiz komşuyu saymak

İki boyutlu bir diziyi `np.roll` ile yukarı, aşağı, sağa ve sola kaydırabiliriz. İki eksende yapılan sekiz farklı kaydırmanın toplamı komşu sayısını verir.

Periyodik sınır kullanıyoruz: üst kenarın komşusu alt kenarda, sol kenarın komşusu sağ kenardadır. Bu, ızgarayı matematiksel olarak bir torus yüzeyine dönüştürür.

### Alıştırma 1

`count_neighbors` fonksiyonunu tamamla. `di` ve `dj` değerlerini `(-1, 0, 1)` üzerinde dolaştır; `(0, 0)` kaydırmasını toplama, çünkü bu merkez hücrenin kendisidir.

In [ ]:
def count_neighbors(grid):
    '''Periyodik sınırlarda her hücrenin sekiz komşusundaki canlıları say.'''
    grid = np.asarray(grid, dtype=np.uint8)
    neighbors = np.zeros_like(grid, dtype=np.uint8)

    # TODO: sekiz kaydırılmış ızgarayı neighbors dizisine ekle
    # İpucu: np.roll(np.roll(grid, di, axis=0), dj, axis=1)

    return neighbors

In [ ]:
def test_count_neighbors():
    grid = np.zeros((5, 5), dtype=np.uint8)
    grid[2, 2] = 1
    result = count_neighbors(grid)
    expected = np.zeros((5, 5), dtype=np.uint8)
    expected[1:4, 1:4] = 1
    expected[2, 2] = 0
    assert np.array_equal(result, expected), "merkezdeki tek hücrenin komşuları yanlış"

    corner = np.zeros((4, 4), dtype=np.uint8)
    corner[0, 0] = 1
    wrapped = count_neighbors(corner)
    assert wrapped[3, 3] == 1 and wrapped[0, 3] == 1 and wrapped[3, 0] == 1, (
        "periyodik sınır köşeler boyunca sarılmalı"
    )
    assert wrapped.sum() == 8, "tek canlı hücre tam sekiz hücreye komşu olmalı"


check("Alıştırma 1 — komşu sayımı", test_count_neighbors)

### Referans komşu sayımı

Büyük deneylerin TODO tamamlanmadan da çalışması için aşağıdaki referans fonksiyonu kullanılacaktır. Testin geçtikten sonra kendi fonksiyonunla aynı sonucu verdiğini karşılaştır.

In [ ]:
def _count_neighbors_reference(grid):
    grid = np.asarray(grid, dtype=np.uint8)
    neighbors = np.zeros_like(grid, dtype=np.uint8)
    for di in (-1, 0, 1):
        for dj in (-1, 0, 1):
            if di == 0 and dj == 0:
                continue
            neighbors += np.roll(np.roll(grid, di, axis=0), dj, axis=1)
    return neighbors

## 2. B3/S23 güncellemesini uygulamak

Önce komşu sayısını hesapla. Sonra iki Boole maskesi oluştur:

```python
birth = (grid == 0) & (neighbors == 3)
survival = (grid == 1) & ((neighbors == 2) | (neighbors == 3))
```

Yeni durum, `birth | survival` ifadesinin 0/1 dizisine dönüştürülmesidir.

### Alıştırma 2

`update_life` fonksiyonunu tamamla. Bu alıştırmada kendi `count_neighbors` fonksiyonunu kullan.

In [ ]:
def update_life(grid):
    '''Game of Life ızgarasını bir zaman adımı ilerlet.'''
    grid = np.asarray(grid, dtype=np.uint8)
    neighbors = count_neighbors(grid)
    # TODO 1: doğum maskesini oluştur
    birth = np.zeros_like(grid, dtype=bool)
    # TODO 2: hayatta kalma maskesini oluştur
    survival = np.zeros_like(grid, dtype=bool)
    return (birth | survival).astype(np.uint8)

In [ ]:
def test_update_life():
    isolated = np.zeros((7, 7), dtype=np.uint8)
    isolated[3, 3] = 1
    assert update_life(isolated).sum() == 0, "izole hücre bir adımda ölmeli"

    block = np.zeros((7, 7), dtype=np.uint8)
    block[2:4, 2:4] = 1
    assert np.array_equal(update_life(block), block), "2×2 blok sabit kalmalı"

    birth_case = np.zeros((7, 7), dtype=np.uint8)
    birth_case[2, 3] = birth_case[3, 2] = birth_case[3, 4] = 1
    assert update_life(birth_case)[3, 3] == 1, "üç komşulu merkez hücre doğmalı"


check("Alıştırma 2 — B3/S23 güncellemesi", test_update_life)

### Referans güncelleme fonksiyonu

In [ ]:
def _update_life_reference(grid):
    grid = np.asarray(grid, dtype=np.uint8)
    neighbors = _count_neighbors_reference(grid)
    birth = (grid == 0) & (neighbors == 3)
    survival = (grid == 1) & ((neighbors == 2) | (neighbors == 3))
    return (birth | survival).astype(np.uint8)

## 3. Çok adımlı benzetim

Bir Game of Life geçmişinin biçimi

```text
(zaman, satır, sütun)
```

olacaktır. İlk katman başlangıç ızgarasıdır.

### Alıştırma 3

`simulate_life` fonksiyonunu tamamla. Her yeni ızgarayı bir önceki ızgaradan hesapla.

In [ ]:
def simulate_life(initial_grid, steps):
    '''Başlangıç durumu dâhil Game of Life geçmişini döndür.'''
    initial_grid = np.asarray(initial_grid, dtype=np.uint8)
    history = np.zeros((steps, *initial_grid.shape), dtype=np.uint8)
    history[0] = initial_grid
    for t in range(1, steps):
        # TODO: update_life kullanarak history[t] değerini hesapla
        history[t] = history[t - 1]
    return history

In [ ]:
def test_simulate_life():
    block = np.zeros((7, 7), dtype=np.uint8)
    block[2:4, 2:4] = 1
    history = simulate_life(block, 5)
    assert history.shape == (5, 7, 7), f"beklenen biçim (5, 7, 7), bulunan {history.shape}"
    assert np.all(history == block), "blok bütün zamanlarda aynı kalmalı"

    isolated = np.zeros((7, 7), dtype=np.uint8)
    isolated[3, 3] = 1
    isolated_history = simulate_life(isolated, 3)
    assert isolated_history[0].sum() == 1 and isolated_history[1:].sum() == 0, (
        "izole hücre başlangıçtan sonra sönmeli"
    )


check("Alıştırma 3 — çok adımlı benzetim", test_simulate_life)

### Referans benzetim fonksiyonu

In [ ]:
def _simulate_life_reference(initial_grid, steps):
    initial_grid = np.asarray(initial_grid, dtype=np.uint8)
    history = np.zeros((steps, *initial_grid.shape), dtype=np.uint8)
    history[0] = initial_grid
    for t in range(1, steps):
        history[t] = _update_life_reference(history[t - 1])
    return history

## 4. Bilinen yapıları hazırlamak

Aşağıdaki yardımcı fonksiyon, koordinat listesiyle verilen küçük bir deseni büyük bir ızgaraya yerleştirir. Koordinatlar `(satır, sütun)` biçimindedir.

In [ ]:
def place_pattern(shape, coordinates, top_left=(0, 0)):
    '''Koordinat listesini verilen sol-üst konumdan başlayarak ızgaraya yerleştir.'''
    grid = np.zeros(shape, dtype=np.uint8)
    row0, col0 = top_left
    for row, col in coordinates:
        grid[row0 + row, col0 + col] = 1
    return grid


PATTERNS = {
    "Blok": [(0, 0), (0, 1), (1, 0), (1, 1)],
    "Arı kovanı": [(0, 1), (0, 2), (1, 0), (1, 3), (2, 1), (2, 2)],
    "Blinker": [(0, 0), (0, 1), (0, 2)],
    "Toad": [(0, 1), (0, 2), (0, 3), (1, 0), (1, 1), (1, 2)],
    "Glider": [(0, 1), (1, 2), (2, 0), (2, 1), (2, 2)],
}

initial_patterns = {
    name: place_pattern((25, 25), coordinates, top_left=(10, 10))
    for name, coordinates in PATTERNS.items()
}

fig, axes = plt.subplots(1, 5, figsize=(13, 3), constrained_layout=True)
for ax, (name, grid) in zip(axes, initial_patterns.items()):
    ax.imshow(grid, interpolation="nearest", vmin=0, vmax=1)
    ax.set_title(name)
    ax.set_xticks([])
    ax.set_yticks([])
plt.show()

### Yapıları çalıştırmadan önce sınıflandır

| Yapı | Sabit / periyodik / hareketli tahminim | Tahminimin gerekçesi |
|---|---|---|
| Blok | | |
| Arı kovanı | | |
| Blinker | | |
| Toad | | |
| Glider | | |

<details>
<summary>Soru: Sabit bir yapıda hücreler hiç güncellenmiyor mu?</summary>

Hücreler her adımda aynı B3/S23 hesabını yapar. Fakat doğum ve ölüm koşulları, bütün desenin bir önceki adımla aynı çıkmasını sağlar. “Sabit” olmak güncellemenin durması değil, güncelleme sonucunun değişmemesidir.
</details>

## 5. Sabit yapılar, salınıcılar ve hareketli yapılar

Her yapı için ilk beş zamanı yan yana gösterelim. Referans motoru kullanıldığı için bu şekil, TODO hücreleri tamamlanmadan da çalışır.

In [ ]:
pattern_histories = {
    name: _simulate_life_reference(grid, 12)
    for name, grid in initial_patterns.items()
}

frames = [0, 1, 2, 3, 4]
fig, axes = plt.subplots(len(PATTERNS), len(frames), figsize=(12, 11), constrained_layout=True)

for row, (name, history) in enumerate(pattern_histories.items()):
    for col, t in enumerate(frames):
        ax = axes[row, col]
        ax.imshow(history[t], interpolation="nearest", vmin=0, vmax=1)
        if row == 0:
            ax.set_title(f"t={t}")
        if col == 0:
            ax.set_ylabel(name)
        ax.set_xticks([])
        ax.set_yticks([])

fig.suptitle("Game of Life yapılarının ilk beş adımı", fontsize=15)
plt.show()

### Gözlemler

Şekilden yararlanarak tabloyu doldur:

| Yapı | Nüfus | Periyot | Periyot başına yer değiştirme | Tür |
|---|---:|---:|---|---|
| Blok | | | | |
| Arı kovanı | | | | |
| Blinker | | | | |
| Toad | | | | |
| Glider | | | | |

“Tür” sütununda **still life**, **oscillator** veya **spaceship** terimlerinden birini kullan.

<details>
<summary>Terminoloji</summary>

- **Still life:** Bir güncellemeden sonra tamamen aynı kalan desen.
- **Oscillator:** Belirli sayıda adım sonra aynı konum ve yönelimde tekrar eden desen.
- **Spaceship:** Belirli sayıda adım sonra biçimini/yönelimini tekrar kazanırken başka bir konuma taşınan desen.
</details>

## 6. Nüfus zaman serisi

Canlı hücre sayısı

\[
N_{\mathrm{alive}}(t)=\sum_{i,j}s_{ij}(t)
\]

bir yapının büyüyüp küçüldüğünü gösterir. Fakat sabit nüfus, sabit desen anlamına gelmez.

### Alıştırma 4

`population_time_series` fonksiyonunu tamamla. Uzay eksenleri olan `axis=(1, 2)` boyunca toplam al.

In [ ]:
def population_time_series(history):
    '''Her zamandaki canlı hücre sayısını döndür.'''
    history = np.asarray(history)
    # TODO: iki uzay ekseni boyunca toplam al
    return np.zeros(history.shape[0], dtype=int)

In [ ]:
def test_population_time_series():
    history = np.array([
        [[0, 0], [0, 0]],
        [[1, 0], [0, 0]],
        [[1, 1], [1, 0]],
    ])
    result = population_time_series(history)
    expected = np.array([0, 1, 3])
    assert np.array_equal(result, expected), f"beklenen {expected}, bulunan {result}"


check("Alıştırma 4 — nüfus", test_population_time_series)

### Referans nüfus ölçümü ve şekil

In [ ]:
def _population_reference(history):
    return np.asarray(history).sum(axis=(1, 2))


fig, ax = plt.subplots(figsize=(9, 4))
for name, history in pattern_histories.items():
    ax.plot(_population_reference(history), marker="o", markersize=3, label=name)

ax.set(
    xlabel="Zaman adımı",
    ylabel="Canlı hücre sayısı",
    title="Bilinen yapıların nüfus zaman serileri",
)
ax.grid(alpha=0.25)
ax.legend(ncol=3)
plt.show()

<details>
<summary>Soru: Blinker ile blok aynı nüfusa sahip olsaydı nüfus grafiği onları ayırabilir miydi?</summary>

Hayır. Nüfus yalnızca toplam canlı hücre sayısını bilir, hücrelerin konumlarını bilmez. Sabit nüfuslu bir oscillator her adımda biçim değiştirirken nüfus grafiği düz bir çizgi olabilir.
</details>

## 7. Periyot algılama

Bir geçmişin son durumu, \(p\) adım önceki durumla aynıysa \(p\) olası bir periyottur. En küçük pozitif \(p\)'yi arayacağız.

Bu basit yöntem geçici davranışları bütünüyle analiz etmez; yalnızca geçmişin sonunda bir tekrar arar.

### Alıştırma 5

`detect_period` fonksiyonunu tamamla. `1`'den `max_period` değerine kadar dene ve son ızgarayı `period` adım önceki ızgarayla karşılaştır. Eşleşme yoksa `None` döndür.

In [ ]:
def detect_period(history, max_period=10):
    '''Geçmişin sonundaki en küçük tekrar periyodunu veya None döndür.'''
    history = np.asarray(history)
    largest = min(max_period, len(history) - 1)
    # TODO: period=1...largest için son durumu karşılaştır
    return None

In [ ]:
def test_detect_period():
    a = np.array([[0, 0], [0, 0]])
    b = np.array([[1, 0], [0, 0]])
    c = np.array([[0, 1], [0, 0]])

    fixed = np.array([a, a, a])
    alternating = np.array([a, b, a, b])
    no_repeat = np.array([a, b, c])

    assert detect_period(fixed, 4) == 1, "sabit geçmişin periyodu 1 olmalı"
    assert detect_period(alternating, 4) == 2, "alternatif geçmişin periyodu 2 olmalı"
    assert detect_period(no_repeat, 4) is None, "tekrar yoksa None dönmeli"


check("Alıştırma 5 — periyot", test_detect_period)

### Referans periyot ölçümü

Glider, aynı biçimi başka konumda tekrarladığı için bu basit fonksiyon onu periyodik olarak tanımaz. Bu bir hata değil: fonksiyon **aynı konumdaki tam eşitliği** ölçmektedir.

In [ ]:
def _detect_period_reference(history, max_period=10):
    largest = min(max_period, len(history) - 1)
    for period in range(1, largest + 1):
        if np.array_equal(history[-1], history[-1 - period]):
            return period
    return None


for name, history in pattern_histories.items():
    print(f"{name:12s}: aynı-konum periyodu = {_detect_period_reference(history)}")

## 8. Gliderın konumu ve hızı

Canlı hücrelerin ortalama satır ve sütun koordinatlarını gliderın yaklaşık merkezi olarak tanımlayalım:

\[
\mathbf r_{\mathrm{cm}}(t)
=\frac{1}{N_{\mathrm{alive}}(t)}
\sum_{(i,j)\,\mathrm{canlı}}(i,j).
\]

Glider bir periyot içinde biçim değiştirdiği için bu merkez hafifçe “sallanabilir”. Dört adım arayla karşılaştırdığımızda temiz bir yer değiştirme elde ederiz.

### Alıştırma 6

`live_centroid` fonksiyonunu tamamla. `np.argwhere(grid == 1)` canlı koordinatları verir. Canlı hücre yoksa `(np.nan, np.nan)` döndür.

In [ ]:
def live_centroid(grid):
    '''Canlı hücrelerin ortalama (satır, sütun) koordinatını döndür.'''
    positions = np.argwhere(np.asarray(grid) == 1)
    # TODO: boş durumu ele al; değilse iki koordinatın ortalamasını döndür
    return (np.nan, np.nan)

In [ ]:
def test_live_centroid():
    grid = np.zeros((6, 7), dtype=np.uint8)
    grid[1, 2] = 1
    grid[3, 4] = 1
    result = live_centroid(grid)
    assert np.allclose(result, (2.0, 3.0)), f"beklenen (2.0, 3.0), bulunan {result}"

    empty = live_centroid(np.zeros((4, 4), dtype=np.uint8))
    assert np.isnan(empty[0]) and np.isnan(empty[1]), "boş ızgaranın merkezi nan olmalı"


check("Alıştırma 6 — canlı hücre merkezi", test_live_centroid)

### Glider deneyinin referans ölçümü

Gliderı kenarlardan uzağa yerleştirip 21 adım çalıştıracağız. Böylece periyodik sınırdan geçmediği için sıradan koordinat ortalaması güvenlidir.

In [ ]:
def _live_centroid_reference(grid):
    positions = np.argwhere(np.asarray(grid) == 1)
    if len(positions) == 0:
        return (np.nan, np.nan)
    return tuple(positions.mean(axis=0))


glider_initial = place_pattern((40, 40), PATTERNS["Glider"], top_left=(5, 5))
glider_history = _simulate_life_reference(glider_initial, 21)
centroids = np.array([_live_centroid_reference(frame) for frame in glider_history])

fig, (ax_path, ax_displacement) = plt.subplots(1, 2, figsize=(11, 4.5), constrained_layout=True)

ax_path.plot(centroids[:, 1], centroids[:, 0], "o-", markersize=3)
ax_path.invert_yaxis()
ax_path.set(
    xlabel="Sütun merkezi",
    ylabel="Satır merkezi",
    title="Glider merkezinin iki boyutlu yolu",
)
ax_path.grid(alpha=0.25)

displacement = centroids - centroids[0]
ax_displacement.plot(displacement[:, 0], "o-", markersize=3, label="Satır yönü")
ax_displacement.plot(displacement[:, 1], "s-", markersize=3, label="Sütun yönü")
ax_displacement.set(
    xlabel="Zaman adımı",
    ylabel="Başlangıçtan yer değiştirme",
    title="Gliderın bileşenler boyunca yer değiştirmesi",
)
ax_displacement.grid(alpha=0.25)
ax_displacement.legend()
plt.show()

shifted_after_four = np.roll(np.roll(glider_history[0], 1, axis=0), 1, axis=1)
print("t=4 deseni, t=0 deseninin (1,1) ötelenmiş hâli mi?",
      np.array_equal(glider_history[4], shifted_after_four))
print("Dört adımlık yer değiştirme:", centroids[4] - centroids[0])
print("Ortalama hız (hücre/adım):", (centroids[4] - centroids[0]) / 4)

### Glider ölçümünü yorumla

1. Dört adımdan sonra desenin yönelimi ve biçimi başlangıçla nasıl ilişkilidir?
2. Dört adımlık yer değiştirme nedir?
3. Satır ve sütun yönlerindeki ortalama hız bileşenleri nelerdir?
4. Gliderın nüfusu sabit midir? Bu, hücrelerin tek tek sabit kaldığı anlamına gelir mi?

**Yanıtlarım:**

> Buraya 6–10 cümle yaz ve en az bir sayısal ölçüm kullan.

<details>
<summary>Soru: Glider gerçekten aynı beş hücrenin hareket etmesi midir?</summary>

Hayır. Izgara hücreleri sabit konumlardadır. Bazı hücreler ölürken başka konumlardakiler doğar. Dört adımlık doğum ve ölüm dizisi, aynı büyük ölçekli biçimin bir hücre çaprazda yeniden oluşmasına yol açar. Hareket eden şey tek tek hücreler değil, desendir.
</details>

## 9. Glider neden beliren bir nesnedir?

Mikroskobik model yalnızca şunları içerir:

- sabit ızgara noktaları;
- 0/1 durumları;
- sekiz komşunun sayısı;
- B3/S23 güncellemesi.

Fakat daha büyük ölçekte gliderı şu özelliklerle tanımlarız:

- konum;
- yön;
- periyot;
- hız;
- yaşam süresi;
- başka desenlerle çarpışma yeteneği.

Bu özellikler tek bir hücrenin kuralında yazılı değildir. Birçok hücrenin örgütlü evriminden ortaya çıkar. Fizikte buna benzer biçimde, mikroskobik bileşenlerden kolektif veya etkin nesneler oluşabilir.

### Kısa sorular

1. Gliderı “parçacık” olarak adlandırmak hangi açılardan yararlıdır?
2. Bu benzetme hangi açılardan eksiktir?
3. Gliderın hızını değiştirmek için tek bir hücrenin hızını değiştirebilir miyiz?
4. Game of Life deterministik olduğu hâlde karmaşık çarpışmalar nasıl oluşabilir?

**Yanıtlarım:**

> Her soruya 2–4 cümleyle yanıt ver.

## 10. İsteğe bağlı araştırma: glidera tek-hücre pertürbasyonu

3. haftayla doğrudan bağlantı kur:

1. Normal glider ızgarasını `A(0)` olarak al.
2. Gliderın yakınındaki bir hücreyi değiştirerek `B(0)` oluştur.
3. İki sistemi 30–50 adım ilerlet.
4. Fark alanını ve Hamming uzaklığını hesapla.
5. Pertürbasyon gliderı yok ediyor, yönünü değiştiriyor veya yeni yapılar oluşturuyor mu?

Bir hücreyi gliderın **içinde** ve **uzağında** değiştirerek iki deneyi karşılaştır. Periyodik sınıra ulaşmayacak kadar büyük bir ızgara kullan.

In [ ]:
# İSTEĞE BAĞLI: Glider pertürbasyonu deneyini burada gerçekleştir.
# En az bir şekil ve kısa bir markdown yorumu ekle.

## 11. Sonuç ve haftalık kontrol noktası

### Ana sonuç tablosu

| Yapı | Nüfus davranışı | Periyot | Hareket | Beliren özellik |
|---|---|---:|---|---|
| Blok | | | | |
| Arı kovanı | | | | |
| Blinker | | | | |
| Toad | | | | |
| Glider | | | | |

### Çıkış sorusu

Aşağıdaki soruya 7–10 cümleyle yanıt ver:

> Game of Life'daki glider neden beliren bir nesne olarak düşünülebilir? Yanıtında **yerel kural**, **mikroskobik hücre**, **büyük ölçekli desen**, **periyot**, **yer değiştirme** ve **etkin hız** kavramlarını kullan.

### Görüşmeye getirilecekler

- altı alıştırma testinin geçmesi;
- deney öncesi tahminler;
- beş yapının ilk beş adımını gösteren şekil;
- doldurulmuş sınıflandırma ve ana sonuç tabloları;
- nüfus grafiği ve periyot ölçümleri;
- glider yolu, yer değiştirmesi ve hız hesabı;
- kısa sorulara verdiğin yanıtlar;
- tartışmak istediğin **iki soru**.

### GitHub teslimi

Dosya adı:

```text
04_game_of_life.ipynb
```

Önerilen kayıt mesajı:

```text
4. Hafta: Conway'in Yaşam Oyunu
```

Teslimden önce `Restart Kernel and Run All Cells` çalıştır. Test çıktılarını ve şekilleri not defterinde bırak.